In [79]:
import joblib
import pandas as pd
import numpy as np
# import the fraud detector pipeline
model_path = "../code/custom-classifier-model/fraud_detector.pkl"
model = joblib.load(model_path)

In [80]:
# Re-load artifact in case the existing model-loading cell loaded a dict wrapper
model_path = "../code/custom-classifier-model/fraud_detector.pkl"
artifact = joblib.load(model_path)

if isinstance(artifact, dict):
    model = artifact.get("pipeline")
    threshold = float(artifact.get("threshold", 0.5))
    feature_columns = artifact.get("feature_columns")
else:
    model = artifact
    threshold = 0.5
    feature_columns = None


In [81]:
def analyze_data(user_input_data, verbose=True, return_details=False):
    """Analyze a single transaction and explain why the model made its decision."""

    def prepare_features(user_df: pd.DataFrame) -> pd.DataFrame:
        """Create the engineered feature columns used during training."""
        dfp = user_df.copy()

        # Make sure numeric inputs are numeric (some come in as strings from the UI/text file)
        numeric_cols = [
            'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
            'oldbalanceDest', 'newbalanceDest'
        ]
        for c in numeric_cols:
            dfp[c] = pd.to_numeric(dfp[c], errors='coerce').fillna(0)

        dfp['type'] = dfp['type'].astype(str)
        dfp['nameOrig'] = dfp['nameOrig'].astype(str)
        dfp['nameDest'] = dfp['nameDest'].astype(str)

        dfp['nameOrig_prefix'] = dfp['nameOrig'].str[0]
        dfp['nameDest_prefix'] = dfp['nameDest'].str[0]

        dfp['orig_balance_delta'] = dfp['newbalanceOrig'] - dfp['oldbalanceOrg']
        dfp['dest_balance_delta'] = dfp['newbalanceDest'] - dfp['oldbalanceDest']

        old_org_denom = dfp['oldbalanceOrg'].replace(0, np.nan).fillna(1)
        new_org_denom = dfp['newbalanceOrig'].replace(0, np.nan).fillna(1)

        dfp['amount_to_oldOrg'] = dfp['amount'] / old_org_denom
        dfp['amount_to_newOrg'] = dfp['amount'] / new_org_denom

        return dfp[feature_columns]

    def neutralize_feature_value(value):
        """Build a no-signal baseline value for a feature."""
        if isinstance(value, str):
            return 'UNKNOWN'
        if pd.api.types.is_number(value):
            return 0.0
        return 0

    def compute_feature_impacts(features_one_row: pd.DataFrame, base_proba: float):
        """Estimate per-feature impact by replacing one feature with a neutral baseline."""
        impacts = []
        for col in features_one_row.columns:
            modified = features_one_row.copy()
            modified_value = neutralize_feature_value(features_one_row.iloc[0][col])
            modified.loc[modified.index[0], col] = modified_value
            modified_proba = float(model.predict_proba(modified)[0, 1])
            delta = base_proba - modified_proba
            impacts.append({
                'feature': col,
                'value': features_one_row.iloc[0][col],
                'neutral_value': modified_value,
                'impact': delta,
            })
        return impacts

    def build_explanation(impacts, top_k=4):
        """Turn raw impacts into readable, model-based reasons."""

        def describe_feature(feature, value):
            v = value
            if feature == 'type':
                return f"transaction type is '{v}'"
            if feature == 'amount':
                return f"amount is ${v:,.2f}"
            if feature == 'step':
                return f"step number is {v}"
            if feature == 'oldbalanceOrg':
                return f"origin old balance is ${v:,.2f}"
            if feature == 'newbalanceOrig':
                return f"origin new balance is ${v:,.2f}"
            if feature == 'oldbalanceDest':
                return f"destination old balance is ${v:,.2f}"
            if feature == 'newbalanceDest':
                return f"destination new balance is ${v:,.2f}"
            if feature == 'isFlaggedFraud':
                return "historically flagged as fraud" if v == 1 else "not historically flagged as fraud"
            if feature == 'orig_balance_delta':
                return f"origin balance change of ${v:,.2f}"
            if feature == 'dest_balance_delta':
                return f"destination balance change of ${v:,.2f}"
            if feature == 'amount_to_oldOrg':
                return f"amount relative to old origin balance is {v:.2f}x"
            if feature == 'amount_to_newOrg':
                return f"amount relative to new origin balance is {v:.2f}x"
            if feature == 'nameOrig_prefix':
                return f"origin name starts with '{v}'"
            if feature == 'nameDest_prefix':
                return f"destination name starts with '{v}'"
            return f"{feature} is {v}"

        sorted_impacts = sorted(impacts, key=lambda x: abs(x['impact']), reverse=True)
        fraud_drivers = [x for x in sorted_impacts if x['impact'] > 0][:top_k]
        nonfraud_drivers = [x for x in sorted_impacts if x['impact'] < 0][:top_k]

        fraud_lines = [
            f"{describe_feature(x['feature'], x['value']).capitalize()} increased fraud likelihood by {x['impact']:.4f}."
            for x in fraud_drivers
        ]
        nonfraud_lines = [
            f"{describe_feature(x['feature'], x['value']).capitalize()} reduced fraud likelihood by {abs(x['impact']):.4f}."
            for x in nonfraud_drivers
        ]

        if not fraud_lines:
            fraud_lines = ["No model features were found that meaningfully increased fraud likelihood."]
        if not nonfraud_lines:
            nonfraud_lines = ["No model features were found that meaningfully reduced fraud likelihood."]

        summary = (
            "Highlights: " + " ".join(fraud_lines + nonfraud_lines)
        )

        return {
            'fraud_drivers': fraud_lines,
            'nonfraud_drivers': nonfraud_lines,
            'summary': summary,
        }

    # ensure dataframe exists
    if isinstance(user_input_data, dict):
        # if values are scalars, wrap them in a single row
        if not any(isinstance(v, (list, tuple, pd.Series)) for v in user_input_data.values()):
            user_df = pd.DataFrame([user_input_data])
        else:
            user_df = pd.DataFrame(user_input_data)
    elif isinstance(user_input_data, pd.Series):
        user_df = user_input_data.to_frame().T
    else:
        user_df = user_input_data

    expected_columns = [
        'step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg',
        'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'
    ]
    user_df = user_df[expected_columns]

    details = {
        'prediction': None,
        'fraud_probability': None,
        'threshold': threshold,
        'explanation_summary': 'Explanation unavailable for this model format.',
        'fraud_drivers': [],
        'nonfraud_drivers': [],
    }

    if feature_columns is not None:
        # Predict using engineered features + tuned threshold
        features = prepare_features(user_df)
        proba = float(model.predict_proba(features)[0, 1])
        prediction = int(proba >= threshold)

        impacts = compute_feature_impacts(features, proba)
        explanation = build_explanation(impacts)

        details.update({
            'prediction': prediction,
            'fraud_probability': proba,
            'explanation_summary': explanation['summary'],
            'fraud_drivers': explanation['fraud_drivers'],
            'nonfraud_drivers': explanation['nonfraud_drivers'],
        })
    else:
        # Fallback for older saved models
        prediction = int(model.predict(user_df)[0])
        details['prediction'] = prediction

    if verbose:
        if details['prediction'] == 1:
            print("Financial fraud is detected.")
        else:
            print("No financial fraud detected.")

        if details['fraud_probability'] is not None:
            print(f"Fraud probability: {details['fraud_probability']:.4f} (threshold={details['threshold']:.4f})")
            print("Why this prediction:")
            for line in details['fraud_drivers']:
                print(f"  + {line}")
            for line in details['nonfraud_drivers']:
                print(f"  - {line}")

    if return_details:
        return details

    return details['prediction']


In [82]:
'''
# example user input matching the models expected features
# used for testing purposes
user_input_data = {
    'step': 1,
    'type': 'PAYMENT',
    'amount': 1000.0,
    'nameOrig': 'C123456789',
    'oldbalanceOrg': 5000.0,
    'newbalanceOrig': 4000.0,
    'nameDest': 'M123456789',
    'oldbalanceDest': 0.0,
    'newbalanceDest': 0.0,
    'isFlaggedFraud': 0,
}

# call the analyze_data function
analyze_data(user_input_data)
'''

"\n# example user input matching the models expected features\n# used for testing purposes\nuser_input_data = {\n    'step': 1,\n    'type': 'PAYMENT',\n    'amount': 1000.0,\n    'nameOrig': 'C123456789',\n    'oldbalanceOrg': 5000.0,\n    'newbalanceOrig': 4000.0,\n    'nameDest': 'M123456789',\n    'oldbalanceDest': 0.0,\n    'newbalanceDest': 0.0,\n    'isFlaggedFraud': 0,\n}\n\n# call the analyze_data function\nanalyze_data(user_input_data)\n"

In [83]:
'''
# simple test cases
# not fraud
non_fraud_tx = {
    'step': 1,
    'type': 'PAYMENT',
    'amount': 100.0,
    'nameOrig': 'C000000001',
    'oldbalanceOrg': 1000.0,
    'newbalanceOrig': 900.0,
    'nameDest': 'M000000001',
    'oldbalanceDest': 0.0,
    'newbalanceDest': 0.0,
    'isFlaggedFraud': 0,
}

print("Non-fraud test case:")
non_fraud_pred = analyze_data(non_fraud_tx)
print(f"Model output (0 = no fraud, 1 = fraud): {non_fraud_pred}\n")

# fraud transfer with large amount and zero resulting balance
fraud_like_tx = {
    'step': 1,
    'type': 'TRANSFER',
    'amount': 250000.0,
    'nameOrig': 'C000000002',
    'oldbalanceOrg': 250000.0,
    'newbalanceOrig': 0.0,
    'nameDest': 'C000000003',
    'oldbalanceDest': 0.0,
    'newbalanceDest': 0.0,
    'isFlaggedFraud': 0,
}

print("Fraud-like test case:")
fraud_like_pred = analyze_data(fraud_like_tx)
print(f"Model output (0 = no fraud, 1 = fraud): {fraud_like_pred}")

# fraud transfer with large amount and zero resulting balance
fraud_like_tx2 = {
    'step': 1,
    'type': 'TRANSFER',
    'amount': 90000.00,
    'nameOrig': '67',
    'oldbalanceOrg': 90000.0,
    'newbalanceOrig': 0.00,
    'nameDest': '68',
    'oldbalanceDest': 0.0,
    'newbalanceDest': 90000.0,
    'isFlaggedFraud': 0,
}
print("Fraud-like test case2:")
fraud_like_pred2 = analyze_data(fraud_like_tx2)
print(f"Model output (0 = no fraud, 1 = fraud): {fraud_like_pred2}")
'''

'\n# simple test cases\n# not fraud\nnon_fraud_tx = {\n    \'step\': 1,\n    \'type\': \'PAYMENT\',\n    \'amount\': 100.0,\n    \'nameOrig\': \'C000000001\',\n    \'oldbalanceOrg\': 1000.0,\n    \'newbalanceOrig\': 900.0,\n    \'nameDest\': \'M000000001\',\n    \'oldbalanceDest\': 0.0,\n    \'newbalanceDest\': 0.0,\n    \'isFlaggedFraud\': 0,\n}\n\nprint("Non-fraud test case:")\nnon_fraud_pred = analyze_data(non_fraud_tx)\nprint(f"Model output (0 = no fraud, 1 = fraud): {non_fraud_pred}\n")\n\n# fraud transfer with large amount and zero resulting balance\nfraud_like_tx = {\n    \'step\': 1,\n    \'type\': \'TRANSFER\',\n    \'amount\': 250000.0,\n    \'nameOrig\': \'C000000002\',\n    \'oldbalanceOrg\': 250000.0,\n    \'newbalanceOrig\': 0.0,\n    \'nameDest\': \'C000000003\',\n    \'oldbalanceDest\': 0.0,\n    \'newbalanceDest\': 0.0,\n    \'isFlaggedFraud\': 0,\n}\n\nprint("Fraud-like test case:")\nfraud_like_pred = analyze_data(fraud_like_tx)\nprint(f"Model output (0 = no fraud, 1 

In [84]:
'''
How to use:
1-Locate the input_data.txt file in the folder.
2-Find the data you want to check for fraud on.
3-Enter that data onto input_data.txt. To test for fraud on multiple individual cases of data, separate each line of data with a new line.
3.1-Data is formatted as <step>,<type>,<amount>,<nameOrig>,<oldbalanceOrg>,<newbalanceOrig>,<nameDest>,<oldbalanceDest>,<newbalanceDest>,<isFlaggedFraud>
3.2-<isFlaggedFraud> must be a 0 or a 1. 0 if it is not flagged as fraud, 1 if it is flagged as fraud.
3.2.1-Example of a correctly formatted input: 1,TRANSFER,181.00,C2048537720,170136.0,160296.36,C553264065,0.0,0.0,0
4-Run the program and wait for it to finish running.
5-Locate and open the output_analysis.txt file. This file prints out the given data, shows if it is fraud or not, and explains why.
'''

'\nHow to use:\n1-Locate the input_data.txt file in the folder.\n2-Find the data you want to check for fraud on.\n3-Enter that data onto input_data.txt. To test for fraud on multiple individual cases of data, separate each line of data with a new line.\n3.1-Data is formatted as <step>,<type>,<amount>,<nameOrig>,<oldbalanceOrg>,<newbalanceOrig>,<nameDest>,<oldbalanceDest>,<newbalanceDest>,<isFlaggedFraud>\n3.2-<isFlaggedFraud> must be a 0 or a 1. 0 if it is not flagged as fraud, 1 if it is flagged as fraud.\n3.2.1-Example of a correctly formatted input: 1,TRANSFER,181.00,C2048537720,170136.0,160296.36,C553264065,0.0,0.0,0\n4-Run the program and wait for it to finish running.\n5-Locate and open the output_analysis.txt file. This file prints out the given data, shows if it is fraud or not, and explains why.\n'

In [85]:
# process input_data.txt and write predictions to output_analysis.txt
input_filepath = 'input_data.txt'
output_filepath = 'output_analysis.txt'

# Define expected columns in input file
columns = ['step','type','amount','nameOrig','oldbalanceOrg','newbalanceOrig','nameDest','oldbalanceDest','newbalanceDest','isFlaggedFraud']

records = []
with open(input_filepath, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = [x.strip() for x in line.split(',')]
        if len(parts) != len(columns):
            print(f"Skipping line {line_num}: expected {len(columns)} values, got {len(parts)}")
            continue
        record = dict(zip(columns, parts))
        # cast numeric fields
        try:
            record['step'] = int(record['step'])
            record['amount'] = float(record['amount'])
            record['oldbalanceOrg'] = float(record['oldbalanceOrg'])
            record['newbalanceOrig'] = float(record['newbalanceOrig'])
            record['oldbalanceDest'] = float(record['oldbalanceDest'])
            record['newbalanceDest'] = float(record['newbalanceDest'])
            record['isFlaggedFraud'] = int(record['isFlaggedFraud'])
            if record['isFlaggedFraud'] not in (0, 1):
                raise ValueError('isFlaggedFraud must be 0 or 1')
        except Exception as e:
            print(f"Skipping line {line_num} due to parse error: {e}")
            continue
        records.append(record)

if not records:
    raise ValueError('No valid records found in input_data.txt')

input_df = pd.DataFrame(records)

# run predictions for each record
output_rows = []
for idx, row in input_df.iterrows():
    details = analyze_data(row, verbose=False, return_details=True)

    pred = int(details['prediction'])
    proba = details['fraud_probability']
    threshold_used = details['threshold']

    reason = details['explanation_summary']
    # Keep csv structure stable by removing commas from reason text
    reason = reason.replace(',', ';')

    output_rows.append({
        'line': idx+1,
        'step': row['step'],
        'type': row['type'],
        'amount': row['amount'],
        'isFlaggedFraud': row['isFlaggedFraud'],
        'prediction': pred,
        'fraud_probability': '' if proba is None else f"{proba:.6f}",
        'threshold': f"{threshold_used:.6f}",
        'reason': reason,
        'conclusion': 'There is a 0% chance that fraud occurred.' if proba is None else 'There is a ' + f"{(proba * 10):.6f}" + ' chance that fraud occurred.'
    })

# write analysis output
with open(output_filepath, 'w', encoding='utf-8') as out:
    out.write('line,step,type,amount,isFlaggedFraud,prediction,fraud_probability,threshold,reason,conclusion\n\n')
    for r in output_rows:
        out.write(
            f"{r['line']},{r['step']},{r['type']},{r['amount']},{r['isFlaggedFraud']},"
            f"{r['prediction']},{r['fraud_probability']},{r['threshold']},{r['reason']},{r['conclusion']}\n\n"
        )

print(f'Written {len(output_rows)} records to {output_filepath}')

Written 5 records to output_analysis.txt
